In [2]:
# Neural Network Analysis Demo
# This notebook demonstrates how to use the NetworkAnalyzer and WeightBinning classes

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

# Import our custom classes
from nnanalyzer.src.NetworkAnalyzer import NetworkAnalyzer
from nnanalyzer.src.binningweights import WeightBinning

import torch.nn as nn

#2-5-5-1
#2 inputs, 2 hidden with 5 neurons each
class SimpleNet(nn.Module):
    # 2 because a, b
    input_size = 2
    # num of hidden neurons
    hidden_size = 5
    output_size = 1
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(self.input_size, self.hidden_size)
        # Right now RELU but we can change later
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(self.hidden_size, self.hidden_size)
        self.fc3 = nn.Linear(self.hidden_size, self.output_size)
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

def load_data():
    csv_data = pd.read_csv('/home2/coles8/Large-Scale-Design-and-Analysis-of-Neural-Networks/data/simpleReg.csv')
    a = csv_data['a'].values.reshape(-1, 1)
    b = csv_data['b'].values.reshape(-1, 1)
    y = csv_data['y'].values.reshape(-1, 1)

    a_tensor = torch.tensor(a, dtype=torch.float32)
    b_tensor = torch.tensor(b, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)
    X_tensor = torch.cat((a_tensor, b_tensor), dim=1)

    X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    batch_size = 64
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    return train_loader, test_loader

train_loader, test_loader = load_data()

FileNotFoundError: [Errno 2] No such file or directory: 'data/simpleReg.csv'

In [ ]:

# 3. Initialize the NetworkAnalyzer
# Generate 5 successful networks with loss less than 0.1
analyzer = NetworkAnalyzer(
    model_architecture=SimpleNet,
    amount_to_produce=5,
    success_loss=0.1,
    convergence_threshold=0.001,
    max_attempts=10
)

# 4. Generate networks
print("Generating networks...")
analyzer.generate_networks(
    train_loader=train_loader,
    test_loader=test_loader,
    num_epochs=50,
    loss_fn=nn.MSELoss(),
    learning_rate=0.01
)



In [ ]:

# 6. Initialize the WeightBinning class to analyze networks
print("\nAnalyzing weight distributions...")
weight_analyzer = WeightBinning(
    architecture=SimpleNet,
    save_dir="/home2/coles8/Large-Scale-Design-and-Analysis-of-Neural-Networks/Notebooks/outputs"
)

# 7. Process the networks and analyze their weights
# Load and store the weights from the networks
weight_analyzer.load_models()
weight_distributions, bin_ranges = weight_analyzer.store_weights()

# 8. Normalize distributions for better visualization
normalized_distributions = weight_analyzer.normalize_distributions()

# 9. Fit Gaussian curves to the weight distributions
weight_analyzer.fit()

# 10. Plot weight distributions for a specific weight in the first layer
layer_idx = 0
neuron_idx = 0
weight_idx = 0
print(f"\nPlotting weight distribution for Layer {layer_idx}, Neuron {neuron_idx}, Weight {weight_idx}")
weight_analyzer.plot_weight_bins_with_fit(
    normalized_distributions, 
    bin_ranges[layer_idx], 
    weight_analyzer.fit_params, 
    layer=layer_idx, 
    weight_position=(neuron_idx, weight_idx)
)

# 11. Cluster the Gaussians and find the unique distributions
print("\nClustering weight distributions...")
kl_indices, ce_indices = weight_analyzer.kl_ce_indices()

# 12. Plot unique distributions for the first layer
print("\nPlotting unique weight distributions for Layer 0")
weight_analyzer.plot_unique_distributions(
    kl_indices, 
    layer=0, 
    normalized_distributions=normalized_distributions, 
    all_bin_edges=bin_ranges, 
    fit_params=weight_analyzer.fit_params
)

print("\nAnalysis complete! Check the 'weight_analysis_results' directory for plots.")